# Revision results and manuscript tables

Consume explicitly selected saved results from training, K-Fold, ablation or offline evaluation. This notebook does not retrain models or repeat inference. Use the **rnet** kernel.

Full-volume statistics count biological animals. Repeated model predictions are averaged per scan, then longitudinal scans per animal. Native patch measurements remain pooled cohort results in a separate table. Cost measurements remain per fold or per scan/model.

The additional baseline experiments from step 6 and the separate D4 manual-volume analysis are deferred. This notebook reports available evidence without filling those gaps.


In [ ]:
from pathlib import Path
import sys

REPO_ROOT: Path = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src/experiments/config.py").is_file()
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from IPython.display import display
import pandas as pd
import plotly.express as px
import plotly.io as pio

pio.renderers.default = "vscode"

## Explicit result sources and analysis settings

For one run, use `regions/full_volume_metrics.csv` and supply its variant name and training seed. For a study, use its top-level `full_volume_metrics.csv` with `name=None` to retain existing variant/seed columns. Offline evaluation saves the same long-form contract. Do not load both a study CSV and its individual run CSVs: duplicate measures are rejected.

External nnU-Net results can enter as an explicit CSV with canonical `Subject`, `Scan`, `Dataset`, `Region`, `Metric`, `Value`, `Unit`, `Status` columns and explicit variant/seed/fold/role metadata. Use the same biological IDs, annotations, region semantics and physical units. Missing annotations must be NaN with `Status="unavailable"`.

Validation roles are excluded by default. To summarize historical rat CV, explicitly select `roles=("validation",)` and describe those results as checkpoint-selection validation. Separate cohorts and anatomical regions are never pooled implicitly.

In [ ]:
from src.evaluation.revision import (
    CostSource, MetricSource, RevisionConfig, export_revision, revision_report,
)

METRIC_SOURCES: list[MetricSource] = [
    # MetricSource(path=REPO_ROOT / "results/comparisons/d1_3drnet_vs_unet/full_volume_metrics.csv"),
    # MetricSource(
    #     path=Path(".../regions/full_volume_metrics.csv"), name="3drnet", seed=42,
    #     manifest=Path(".../subjects.csv"),  # Optional scan metadata for older exports.
    # ),
]
analysis = RevisionConfig(
    roles=("test", "holdout", "external"), metrics=("Dice", "HD95", "ASSD", "AVE"),
    bootstrap_samples=2000, confidence=0.95, seed=42,
    failure_dice_threshold=0.5,  # Review candidates only; never excludes scores.
)
MODEL_COMPARISON: tuple[str, str] | None = None  # ("3drnet", "unet")
RUN_ANALYSIS: bool = False
EXPORT_TABLES: bool = False
OUTPUT_DIR: Path = REPO_ROOT / "outputs" / "revision" / "selected_results"
CLASSIFIER_FILES: tuple[Path, ...] = ()  # Saved run summary/study/checkpoint classifier_predictions.csv.
RUN_CLASSIFICATION: bool = False
EXPORT_CLASSIFICATION: bool = False
CLASSIFIER_OUTPUT_DIR: Path = OUTPUT_DIR / "classification"
PATCH_SOURCES: list[MetricSource] = [
    # MetricSource(path=Path(".../summary/patch_metrics.csv"), name="3drnet", seed=42),
]
COST_SOURCES: list[CostSource] = [
    # Single run: metadata.json supplies recorded devices and precision.
    # CostSource(
    #     path=Path(".../summary/fold_timings.csv"), name="3drnet", seed=42,
    #     metadata_file=Path(".../RUN_.../metadata.json"),
    # ),
    # Checkpoint evaluation: use its own recorded runtime, role and workload.
    # CostSource(
    #     path=Path(".../evaluation/inference_timings.csv"),
    #     name="3drnet", seed=42, fold=0, role="test", dataset="D1",
    #     metadata_file=Path(".../evaluation/evaluation.json"),
    # ),
    # Study timings.csv already includes Variant/Seed/fold. Hardware stays unknown
    # unless you select individual run tables with their own metadata_file.
]

print("Selected metric sources:", len(METRIC_SOURCES))
display(pd.DataFrame({"File": [str(source.path) for source in METRIC_SOURCES]}))

## Animal-aware summaries and coverage

Intervals resample animals after the prespecified scan/model aggregation. They condition on the supplied trained models. Missing annotations, undefined surface distances and invalid values are not zero-filled. Finite both-empty Dice values retain the metric's documented convention; inspect their status counts. The summary reports the number of animals with finite measurements for each region/metric.

In [ ]:
report = None
if RUN_ANALYSIS:
    report = revision_report(
        sources=METRIC_SOURCES, config=analysis, comparison=MODEL_COMPARISON,
        patch_sources=PATCH_SOURCES, cost_sources=COST_SOURCES,
    )
    display(report.coverage)
    display(report.statuses)
    display(report.summary)
    display(report.animals)
else:
    print("Select saved result files, then set RUN_ANALYSIS = True.")

## Segmentation performance

Plot one descriptive value per animal. Longitudinal scan counts remain visible; multiple seeds/fold predictions are not shown as additional animals. Use dataset/role filters to compare the intended cohort.

In [ ]:
if report is not None:
    dice = report.animals.loc[report.animals["Metric"] == "Dice"]
    if not dice.empty:
        figure = px.box(
            dice, x="Variant", y="Value", color="Variant", facet_col="Region",
            facet_row="Dataset", points="all", hover_data=["Subject", "DatasetRole", "Scans"],
            title="Animal mean full-volume Dice",
        )
        figure.show()

## Volume agreement from available annotations

Reference and prediction volumes are paired before averaging. Genuine zero reference volumes remain valid; missing annotations are excluded. This is agreement for **animal mean scan volumes**, not longitudinal limits of agreement. At least three independent animals are required. Check the number of annotated scans contributing to each animal.

Normal-theory limits of agreement assume approximately normal differences and homogeneous variance. Inspect the difference/Q-Q diagnostics and proportional-bias estimate. Bootstrap intervals describe sampling uncertainty. Existing paired and variance diagnostics are exploratory.

This section can use existing reference-volume rows. The separate D4 manual-volume import and analysis are deferred; selecting another cohort here does not complete that analysis.


In [ ]:
from src.evaluation.statistics import (
    AgreementConfig, bland_altman_figure, calculate_agreement, difference_distribution_figure,
)

AGREEMENT_VARIANT: str = "3drnet"
AGREEMENT_DATASET: str = "D1"
AGREEMENT_REGION: str = "Lesion"
AGREEMENT_ROLE: str = "test"

if report is not None:
    display(report.agreement)
    if not report.volumes.empty:
        pairs = report.volumes.loc[
            (report.volumes["Variant"] == AGREEMENT_VARIANT)
            & (report.volumes["Dataset"] == AGREEMENT_DATASET)
            & (report.volumes["Region"] == AGREEMENT_REGION)
            & (report.volumes["DatasetRole"] == AGREEMENT_ROLE)
        ]
        display(pairs)
        if len(pairs) >= 3:
            agreement_settings = AgreementConfig(
                unit="mm3", bootstrap_samples=analysis.bootstrap_samples,
                confidence=analysis.confidence, seed=analysis.seed,
            )
            agreement = calculate_agreement(pairs, agreement_settings)
            bland_altman_figure(pairs, agreement, agreement_settings).show()
            difference_distribution_figure(pairs, agreement_settings).show()
        else:
            print("Choose an agreement group with at least three independent animals.")

## Optional paired architectural comparison

Set `MODEL_COMPARISON` before analysis. Each comparison uses scans with finite measurements from both models, then pairs animal means. Positive differences mean candidate minus reference; higher Dice and lower distances/errors have different preferred directions. Review paired animal counts and common-scan coverage alongside the separate model summaries.

Mean-difference intervals resample paired animals. Exploratory Wilcoxon p-values receive Holm adjustment over all tested dataset/role/region/metric groups returned in this analysis. No test is chosen using normality-test results. No comparison is performed with fewer than three paired animals. Hyperparameters and cohorts must have been selected independently of these evaluation results.

In [ ]:
if report is not None:
    display(report.comparisons)

## Auxiliary classification and healthy lesion false positives

Select complete saved classifier decision tables. SHAM=0 and TBI=1. Scan predictions retain the historical majority of patch decisions (>0.5; ties SHAM); the recorded positive patch fraction is not a calibrated probability. Longitudinal scan decisions are combined by majority per animal, again with ties SHAM. Rates and exact 95% intervals count animals, not scans or seeds. Disjoint outer holdouts also have a pooled row; repeated test/external predictions stay per fold.

Classifier decisions never gate lesion segmentation. SHAM lesion volumes and the fraction of healthy animals with any nonzero lesion prediction are separate diagnostics. These diagnostics also apply to no-classifier models. Validation remains checkpoint-selection performance. Older results without classifier records require fresh checkpoint evaluation.


In [ ]:
from src.evaluation.classification import (
    classification_report, load_classifier_predictions, export_classification_tables,
)

classifier_results = None
classifier_scans = pd.DataFrame()
if RUN_CLASSIFICATION:
    classifier_scans = load_classifier_predictions(CLASSIFIER_FILES)
    if classifier_scans.empty:
        raise ValueError("Select saved classifier_predictions.csv files first")
    classifier_scans = classifier_scans.loc[classifier_scans["DatasetRole"].isin(analysis.roles)]
    if classifier_scans.empty:
        raise ValueError("No classifier decisions match the selected cohort roles")
    classifier_results = classification_report(classifier_scans, confidence=analysis.confidence)
    display(classifier_results.summary)
    display(classifier_results.animals)
if report is not None:
    display(report.sham_lesion_summary)
    display(report.sham_lesion_animals)


In [ ]:
import plotly.graph_objects as go

CLASSIFIER_VARIANT: str = "3drnet"
CLASSIFIER_SEED: int = 42
CLASSIFIER_ROLE: str = "test"
CLASSIFIER_SCOPE: str = "fold"  # pooled_holdout uses Fold=-1.
CLASSIFIER_FOLD: int = 0

if classifier_results is not None:
    confusion = classifier_results.confusion
    selected = confusion.loc[
        (confusion["Variant"] == CLASSIFIER_VARIANT)
        & (confusion["Seed"] == CLASSIFIER_SEED)
        & (confusion["DatasetRole"] == CLASSIFIER_ROLE)
        & (confusion["Scope"] == CLASSIFIER_SCOPE)
        & (confusion["Fold"] == CLASSIFIER_FOLD)
    ]
    if selected.empty:
        print("Choose a variant/seed/role/scope/fold from the summary above.")
    else:
        matrix = selected.pivot(index="Reference", columns="Prediction", values="Count")
        counts = matrix.to_numpy()
        heatmap = go.Heatmap(
            z=counts, text=counts, texttemplate="%{text}",
            x=["Predicted SHAM", "Predicted TBI"],
            y=["Reference SHAM", "Reference TBI"], colorscale="Blues",
        )
        figure = go.Figure(heatmap)
        figure.update_layout(title="Animal-level classifier confusion matrix")
        figure.show(renderer="vscode")

if EXPORT_CLASSIFICATION:
    if classifier_results is None:
        raise ValueError("Run classifier analysis before exporting")
    CLASSIFIER_OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
    source_metrics = pd.DataFrame()
    if report is not None:
        source_metrics = report.raw.loc[report.raw["DatasetRole"].isin(analysis.roles)]
    export_classification_tables(
        classifier_scans, source_metrics, CLASSIFIER_OUTPUT_DIR, confidence=analysis.confidence,
    )
    print("Classifier and SHAM tables saved to:", CLASSIFIER_OUTPUT_DIR)


## Computational costs

`COST_SOURCES` accepts saved `fold_timings.csv`, study `timings.csv`, and `inference_timings.csv`. Training seconds, completed epochs, selected epoch and parameter counts remain fold measurements. Per-volume timings retain preprocessing, inference/reconstruction and postprocessing separately. Device warm-up is excluded by the evaluator.

Select individual run timing files with `metadata_file` to attach their recorded device and precision metadata. Use `evaluation.json` for standalone checkpoint evaluation, whose runtime may differ from training. Set `role` and `dataset` for inference timing files that lack those columns. Without it, these fields remain `unknown`. Recorded devices are available physical device names, not measured GPU utilization or GPU model specifications. Match hardware, precision, cohort and workload before comparing speed. Missing peak-memory measurements are not estimated. Descriptive means are not animal confidence intervals.


In [ ]:
if report is not None:
    display(report.costs.summary)
    display(report.costs.measurements)
    if report.costs.measurements.empty:
        print("Select COST_SOURCES and rerun the analysis to include recorded computational costs.")


## Native patch measurements

These values use the compiled training losses and metrics on saved validation/test patches. They are pooled cohort measurements, including selection-validation when present. Keep them separate from postprocessed full-volume Dice and from animal confidence intervals. Use `PATCH_SOURCES` to include them in the same export.


In [ ]:
if report is not None:
    display(report.patch_metrics)


## Failure-case review

Inspect all candidate rows before choosing manuscript examples. Low Dice, empty predictions, nonempty predictions on empty references and unexpected undefined measurements are review reasons, not confirmed diagnoses. Both-empty conventions and unavailable annotations are retained in the coverage tables. No candidate is automatically removed from the results.

Coverage counts observed rows; scans absent from every selected file are not detectable without an expected manifest. Use notebook 09 to inspect the original saved prediction and GT for the selected subject/scan/fold. Record the visual finding and its source when preparing a figure.


In [ ]:
FAILURE_VARIANT: str | None = None
FAILURE_DATASET: str | None = None
FAILURE_REGION: str | None = None
FAILURE_MAX_ROWS: int = 50  # Display only; the export retains every candidate.

if report is not None:
    candidates = report.failure_cases
    filters = {"Variant": FAILURE_VARIANT, "Dataset": FAILURE_DATASET, "Region": FAILURE_REGION}
    for column, selected_value in filters.items():
        if selected_value is not None:
            candidates = candidates.loc[candidates[column] == selected_value]
    display(candidates.head(FAILURE_MAX_ROWS))
    print("Review candidate measurements:", len(candidates))


## Anatomy sets, cohort metadata and structure size

These additional tables separate SHAM/TBI, anatomy-only versus lesion/brain-inclusive scores, and observed annotation coverage. An available-region mean does not validate missing TBI anatomy. Inspect `ExpectedRegions`, `FiniteRegions` and `ContributingRegions` in the scan table before comparing different annotation subsets. Background is excluded from these macros.

`METRIC_SOURCES` can attach an explicit `manifest` CSV with Subject/Scan/Dataset and metadata columns. New exports already include Condition, Modality, Timepoint, Subgroup, Strain and Protocol where known. Optional expert-provided `FusedBoundary` flags can be added to the manifest; they are never inferred from predictions. Unknown metadata stays explicit.

Participant counts come from supplied manifests. Dataset totals and timepoint/sequence strata are separate: adding stratum animal counts can count the same animal repeatedly. Training run workbooks also contain saved split counts.


In [ ]:
if report is not None:
    display(report.reports.region_set_summary)
    display(report.reports.region_coverage)
    display(report.reports.subgroup_summary)
    display(report.reports.participant_flow)
    display(report.reports.evaluation_flow)


### Explore small structures

Compare full-volume scores against the actual reference volume. Hover identifies animal, scan, seed, fold and empty-mask status. Repeated scan/model points are not independent animals. Missing annotations and undefined boundary distances stay absent from the plot and remain in exported tables. No minimum-volume cutoff is tuned.


In [ ]:
from src.evaluation.reporting import structure_size_figure

SIZE_REGION: str = "Ipsi ventricle"
SIZE_METRIC: str = "Dice"  # Dice, HD95, ASSD, Sensitivity or Precision.

if report is not None:
    display(report.reports.structure_size_metrics)
    sizes = report.reports.structure_size_metrics
    if not sizes.empty and SIZE_REGION in set(sizes["Region"]):
        size_figure = structure_size_figure(report.reports, region=SIZE_REGION, metric=SIZE_METRIC)
        size_figure.show(renderer="vscode")
    else:
        print("Choose SIZE_REGION from the available region names.")


## Export auditable manuscript tables

`EXPORT_TABLES=True` writes CSVs and `revision_tables.xlsx` in a fresh folder. Separate sheets contain native patch results, full-volume animal summaries, coverage/status counts, paired volumes, agreement, classifier-independent healthy-lesion diagnostics, model comparisons, cost records and review candidates.

`source_files.csv` records input SHA256 hashes, including selected hardware metadata. Export rejects inputs changed since analysis. `analysis.json` records the analysis settings; `reporting_notes.md` describes the scope. Values retain full numeric precision; unavailable or undefined measurements stay blank. No interpolation sensitivity analysis is included.


In [ ]:
if EXPORT_TABLES:
    if report is None:
        raise ValueError("Run the analysis before exporting")
    saved_folder = export_revision(result=report, output_dir=OUTPUT_DIR, config=analysis)
    print("Tables saved to:", saved_folder)